# FLUX.1 LoRA Training — High VRAM Profile (Original Quality Settings)

Use this notebook when you have **enough RAM/VRAM** (not the free 12GB Colab tier).

For free-tier Colab (12GB RAM / 15GB VRAM), use **`train_flux_lora.ipynb`** instead.

**Recommended hardware**
- GPU: A100, L4 (24GB+), or RTX 4090 (24GB)
- System RAM: 32GB+ recommended
- Colab: **Pro+** with high-RAM runtime

**Profile used here (`high_vram`)**
- Base model: `FLUX.1-dev` (best quality)
- Resolution: `1024`
- Batch size: `1`, LoRA rank: `8`
- Epochs: `10` (480 steps with ~48 images)
- Mixed precision: `bf16`
- Memory flags: gradient checkpointing, 8-bit Adam, latent caching
- **No** 4-bit / `--low_ram` mode

**Before you start**
1. Runtime → Change runtime type → **GPU** (A100 / L4 preferred)
2. Accept licenses on Hugging Face:
   - https://huggingface.co/black-forest-labs/FLUX.1-dev
3. Create a Hugging Face token: https://huggingface.co/settings/tokens
4. Put raw images on Google Drive (see dataset step)

Expected output: `training/output/mystyle_flux_lora.safetensors`

In [ ]:
# Check GPU + system RAM (aim for 24GB+ VRAM, 32GB+ RAM)
!nvidia-smi
!free -h

In [ ]:
# Set your repo URL
REPO_URL = "https://github.com/PariGarg10/ARTGENERATOR.git"
BRANCH = "main"

# Google Drive folder with raw images (0000.png, 0001.png, ...)
DRIVE_RAW_IMAGES = "/content/drive/MyDrive/flux_dataset/raw"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and auto-detect project folder
import os
from pathlib import Path

%cd /content
!rm -rf project artgenerator

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL to your real GitHub repo URL.")

!git clone --branch {BRANCH} {REPO_URL} project

clone_root = Path("/content/project")
work_dir = None
for path in [clone_root / "project", clone_root]:
    if (path / "training" / "train_flux_lora.py").exists():
        work_dir = path
        break

if work_dir is None:
    !ls -la /content/project
    raise FileNotFoundError("Could not find training/train_flux_lora.py")

os.chdir(work_dir)
print("Working directory:", work_dir)
!pwd

In [ ]:
# Install dependencies (diffusers 0.34 stack)
!pip uninstall -y diffusers -q
!pip install -q -r requirements-colab.txt
!pip install -q "peft>=0.15.0"

import diffusers
import peft
from diffusers.training_utils import _collate_lora_metadata
print("diffusers:", diffusers.__version__, "| peft:", peft.__version__)

from pathlib import Path
script = Path("training/_hf_scripts/train_dreambooth_lora_flux.py")
text = script.read_text(encoding="utf-8") if script.exists() else ""
if 'check_min_version("0.34.0")' not in text:
    raise RuntimeError("Wrong training script. Re-clone the repo.")
print("Training script OK")

In [ ]:
# Hugging Face login (required for gated FLUX.1-dev)
from google.colab import userdata
from huggingface_hub import login, whoami
from transformers import CLIPTokenizer

MODEL_ID = "black-forest-labs/FLUX.1-dev"

try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("Logged in via HF_TOKEN secret")
except Exception:
    login()
    print("Logged in via interactive token")

print("HF user:", whoami()["name"])
CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
print("Tokenizer OK for FLUX.1-dev")

## 1) Copy dataset

In [ ]:
!mkdir -p dataset/raw dataset/processed

import shutil
from pathlib import Path

raw_dir = Path("dataset/raw")
drive_raw = Path(DRIVE_RAW_IMAGES)

if drive_raw.exists():
    for item in drive_raw.iterdir():
        if item.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
            shutil.copy2(item, raw_dir / item.name)
    print(f"Copied from Drive: {drive_raw}")
else:
    print(f"Drive folder not found: {drive_raw}")
    print("Upload images to dataset/raw manually.")

print("Raw images:", len(list(raw_dir.glob("*"))))

In [ ]:
# Original quality: 1024x1024 preprocessing
!python dataset/preprocess_dataset.py \
  --input_dir ./dataset/raw \
  --output_dir ./dataset/processed \
  --size 1024

In [ ]:
!python captions/generate_florence_captions.py \
  --image_dir ./dataset/processed \
  --trigger_token mystyle

## 2) Train LoRA (high VRAM profile)

Uses `FLUX.1-dev`, 1024px, rank 8, bf16 — original settings for best quality.
Training may take 1–3 hours depending on GPU.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("GPU cleared before training.")

In [ ]:
import diffusers
from diffusers.training_utils import _collate_lora_metadata
print("diffusers", diffusers.__version__, "OK")

!python training/train_flux_lora.py \
  --dataset_dir ./dataset/processed \
  --output_dir ./training/output \
  --trigger_token mystyle \
  --profile high_vram

In [ ]:
!ls -lah ./training/output

In [ ]:
from pathlib import Path
import shutil

lora_file = Path("training/output/mystyle_flux_lora.safetensors")
drive_out = Path("/content/drive/MyDrive/flux_training_output_high_vram")
drive_out.mkdir(parents=True, exist_ok=True)

if lora_file.exists():
    dest = drive_out / lora_file.name
    shutil.copy2(lora_file, dest)
    print(f"Saved to Drive: {dest}")
else:
    print("LoRA file not found. Check training logs above.")

## 3) After training (local machine)

Download `mystyle_flux_lora.safetensors` from Drive, then run:

```bash
python inference/generate.py \
  --prompt "a cozy cafe in rain" \
  --lora_path ./training/output/mystyle_flux_lora.safetensors \
  --base_model black-forest-labs/FLUX.1-dev \
  --width 1024 --height 1024 \
  --output ./inference/outputs/sample.png
```

Or launch Gradio:

```bash
python gradio_app.py
```

Set **Base Model** in the UI to `black-forest-labs/FLUX.1-dev` for this LoRA.